# PennyLane circuit optimization via pytket

Build a PennyLane QNode, extract the tape, convert to pytket,
optimize, and convert back. Shows cross-framework optimization.

In [ ]:
from pytket import Circuit
from pytket.passes import FullPeepholeOptimise, RemoveRedundancies
from pytket.qenn import tk_to_qenn, qenn_to_tk

import pennylane as qml
import numpy as np

## Build a PennyLane circuit with redundancies

Same pattern as the Qiskit example: redundant `X`, `H`, and
`T`/`Tdg` pairs.

In [ ]:
dev = qml.device("default.qubit", wires=3)

@qml.qnode(dev)
def circuit():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[1, 2])
    qml.PauliX(wires=0)
    qml.PauliX(wires=0)
    qml.Hadamard(wires=1)
    qml.Hadamard(wires=1)
    qml.CNOT(wires=[2, 0])
    qml.T(wires=0)
    qml.adjoint(qml.T)(wires=0)
    return qml.expval(qml.PauliZ(0))

print(qml.draw(circuit)())
tape = circuit.tape
print(f"Operations: {len(tape.operations)}  Depth: {tape.depth}")

## Convert to pytket and optimize

In [ ]:
tk_circ = qenn_to_tk(tape)
print(f"pytket gates: {tk_circ.n_gates}  depth: {tk_circ.depth()}")

FullPeepholeOptimise().apply(tk_circ)
RemoveRedundancies().apply(tk_circ)

print(f"After optimize: gates={tk_circ.n_gates}  depth={tk_circ.depth()}")

## Convert back to PennyLane

In [ ]:
tape_opt = tk_to_qenn(tk_circ)
print(qml.draw(tape_opt)())
print(f"Operations: {len(tape_opt.operations)}  Depth: {tape_opt.depth}")
print(f"\nReduction: {len(tape.operations)} -> {len(tape_opt.operations)} ops")